# Segmentación de clientes para CRM

**Objetivo:** identificar a los clientes interesados en nuevas colecciones, a los recurrentes y a los de mayor valor, y agruparlos en arquetipos para que el equipo de CRM se comunique de forma distinta con cada grupo.

## 1. Carga y vista general

In [3]:
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().parent
DATA = RAIZ / "data"

ARCHIVOS = ["customers", "orders", "order_items", "products", "stores"]

faltantes = [nombre for nombre in ARCHIVOS if not (DATA / f"{nombre}.csv").exists()]
assert not faltantes, f"Faltan archivos en {DATA}: {faltantes}"

customers   = pd.read_csv(DATA / "customers.csv")
orders      = pd.read_csv(DATA / "orders.csv")
order_items = pd.read_csv(DATA / "order_items.csv")
products    = pd.read_csv(DATA / "products.csv")
stores      = pd.read_csv(DATA / "stores.csv")

tablas = {"customers": customers, "orders": orders, "order_items": order_items,
          "products": products, "stores": stores}

for nombre, df in tablas.items():
    print(f"{nombre}: {df.shape[0]} filas x {df.shape[1]} columnas")

customers: 3122 filas x 10 columnas
orders: 22676 filas x 6 columnas
order_items: 48293 filas x 5 columnas
products: 240 filas x 5 columnas
stores: 5 filas x 2 columnas


In [6]:
for nombre, df in tablas.items():
    print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 3122 entries, 0 to 3121
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   customer_id       3122 non-null   str  
 1   email             3122 non-null   str  
 2   telefono          3112 non-null   str  
 3   fecha_nacimiento  3030 non-null   str  
 4   ciudad            3122 non-null   str  
 5   fecha_registro    3122 non-null   str  
 6   canal_registro    3122 non-null   str  
 7   opt_in_email      3122 non-null   bool 
 8   opt_in_whatsapp   3122 non-null   bool 
 9   notas_crm         165 non-null    str  
dtypes: bool(2), str(8)
memory usage: 201.4 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 22676 entries, 0 to 22675
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   order_id     22676 non-null  str  
 1   customer_id  21876 non-null  str  
 2   fecha        22676 non-null  str  
 3   canal     

**Hallazgos de la estructura**

Se cargaron 5 tablas: 3.122 clientes, 22.676 órdenes, 48.293 ítems de órdenes, 240 productos y 5 tiendas.

1. costumers trae una columna adicional llamada notas_crm
2. Las columnas numéricas (cantidad, precio_unitario, descuento_pct, precio_lista) están bien tipadas como enteros. Que el tipo sea correcto no garantiza que los valores lo sean: se revisan rangos más adelante.
3. descuento_pct es entero lo que sugiere una escala de 0-100, hay que confirmarlo en la siguiente sección
4. las fechas son texto, deben convertirse un revisar si hay formatos mezclados


## 2. Calidad de datos
### 2.1 Duplicados y valores nulos

In [5]:
for nombre, df in tablas.items():
    print(f"\n{nombre}")
    print("  filas duplicadas completas:", df.duplicated().sum())
    print("  nulos por columna:")
    print(df.isna().sum()[df.isna().sum() > 0].to_string())

ids = {"customers": "customer_id", "orders": "order_id",
       "products": "sku", "stores": "tienda_id"}

print("\nIDs repetidos (deberían ser únicos):")
for nombre, col in ids.items():
    print(f"  {nombre}.{col}: {tablas[nombre][col].duplicated().sum()}")


customers
  filas duplicadas completas: 0
  nulos por columna:
telefono              10
fecha_nacimiento      92
notas_crm           2957

orders
  filas duplicadas completas: 0
  nulos por columna:
customer_id      800
tienda_id      12143

order_items
  filas duplicadas completas: 627
  nulos por columna:
Series([], )

products
  filas duplicadas completas: 0
  nulos por columna:
fecha_lanzamiento    24

stores
  filas duplicadas completas: 0
  nulos por columna:
Series([], )

IDs repetidos (deberían ser únicos):
  customers.customer_id: 0
  orders.order_id: 0
  products.sku: 0
  stores.tienda_id: 0


**Hallazgos de duplicados y nulos**

| Tabla | Hallazgo | Hipótesis / impacto |
|---|---|---|
| customers | telefono (10) y fecha_nacimiento (92) nulos | Bajo impacto: no se usan para responder las tres preguntas, sin embargo sirven para revisar la integridad de otras columnas |
| customers | notas_crm vacía en 2.957 de 3.122 clientes (95%) | Solo 165 clientes tienen notas sin embargo algunas de las notas nos dan información que puede aportar de manera significativa al analisis |
| orders | 800 ordenes sin customer_id | Son ventas reales (tienen order_id), pero no se pueden atribuir a un cliente. Cuentan en el total de ventas, no en el análisis por cliente. |
| orders | 12.143 órdenes sin tienda_id | Probablemente son las órdenes online, que no tienen tienda física. |
| order_items | 627 filas duplicadas completas | Posible doble carga. Si no se corrige, infla las ventas y el valor de algunos clientes. |
| products | 24 productos sin fecha_lanzamiento | Verificado: todos son BASICOS, que no tienen temporada. No es un error. |


No hay IDs repetidos en ninguna tabla. Sin embargo, un ID único no garantiza que cada cliente sea una persona distinta: se revisa por email tambien

### 2.2 Plan de verificación

### Hallazgos iniciales y plan de verificación

| Código | Tabla | Análisis | Qué se revisa | Acción si se confirma |
|---|---|---|---|---|
| C1 | customers | Duplicados por email | Mismo email (normalizado) con distinto customer_id | Unificar al cliente más antiguo y reasignar sus órdenes |
| C2 | customers | Formato de email | Emails sin estructura válida | Marcar como no contactable por email |
| C3 | customers | Consistencia de texto | ciudad y canal_registro escritos de varias formas | Normalizar (minúsculas, espacios, sinónimos) |
| C4 | customers | Fechas de cliente | Formatos mezclados, registro futuro, edad < 15 o > 100 | Convertir; los valores imposibles pasan a nulo |
| C5 | customers | Consentimiento vs. contacto | Opt-in activo sin teléfono o con email inválido | Marcar como no activable en ese canal |
| C6 | customers | Contenido de notas_crm | Etiquetas útiles vs. textos con forma de instrucción | Etiquetas como contexto; los lineamientos se validan con el negocio antes de usarlos |
| O1 | orders | Consistencia de texto | canal y estado | Normalizar si hay variantes |
| O2 | orders | Canal vs. tienda | Tienda sin tienda_id u online con tienda_id | Corregir o documentar |
| O3 | orders | Integridad referencial | customer_id que no existe en customers, tienda_id que no existe en stores | Excluir del análisis por cliente; se mantienen en el total |
| O4 | orders | Fechas de orden | Formatos, rango, fechas futuras, órdenes antes del registro | Convertir; documentar los casos imposibles |
| O5 | orders | Distribución de estados | % de completadas, devueltas y canceladas | Solo las completadas cuentan como venta |
| O6 | orders | Ventas por tienda (descriptivo) | Participación de cada tienda y del canal online | Contexto de negocio |
| I1 | order_items | Duplicados exactos | 627 filas idénticas en todas las columnas | Pendiente de confirmar con el negocio; se calcula el impacto con y sin ellas |
| I2 | order_items | Rangos numéricos | cantidad ≤ 0, precio_unitario ≤ 0, descuentos irreales | Excluir o corregir según el caso |
| I3 | order_items | Integridad referencial | order_id que no existe en orders, sku que no existe en products | Excluir los ítems huérfanos |
| I4 | order_items | Precio vs. precio de lista | Precios muy alejados de precio_lista | Revisar si es un error de escala |
| P1 | products | Fechas de lanzamiento nulas | 24 productos sin fecha | Verificado: todos son BASICOS, sin temporada. No es un error. |
| P2 | products | Consistencia de colección | Nombre de colección, categoría y fecha de lanzamiento coherentes con la temporada | Normalizar y documentar las incoherencias |
| X1 | entre tablas | Órdenes sin ítems | Órdenes que no tienen ninguna línea en order_items | Documentar; no generan venta |